# Практика 5. Деревья: подбор параметров и оценка качества

**Курс «Машинное обучение» · Часть 1, занятие 5**

Дерево — первая модель курса, у которой качество целиком зависит от настройки. Из коробки оно почти всегда переобучается, поэтому сегодня учимся подбирать параметры и честно мерить результат.

План:

1. Данные и точка отсчёта
2. Дерево из коробки: откуда берётся переобучение
3. Как глубина влияет на качество — видно по кривым
4. Подбор параметров через `GridSearchCV`
5. Когда параметров много: случайный поиск и подбор по шагам
6. Оценка качества: почему одной accuracy мало
7. Метод соседей и масштаб — короткая врезка

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, make_moons
from scipy.stats import randint
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("готово")

---
## 1. Данные и точка отсчёта

Берём встроенный датасет о диагностике опухолей: 569 объектов, 30 числовых признаков, целевая переменная — доброкачественная опухоль или нет.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

print(f"обучающая: {X_train.shape}, тестовая: {X_test.shape}")
print(f"доля положительного класса: {y.mean():.3f}")

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"accuracy константной модели: {dummy.score(X_test, y_test):.4f}")

---
## 2. Дерево из коробки

Обучим дерево без единого ограничения и посмотрим на разрыв между обучением и тестом.

In [ ]:
tree_default = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(X_train, y_train)

print(f"глубина выросшего дерева: {tree_default.get_depth()}")
print(f"листьев:                  {tree_default.get_n_leaves()}")
print()
print(f"accuracy на обучении: {tree_default.score(X_train, y_train):.4f}")
print(f"accuracy на тесте:    {tree_default.score(X_test, y_test):.4f}")
print()
print("Единица на обучении — это не успех, а признак того, что дерево")
print("дорастило ветви до отдельных объектов.")

In [ ]:
# Дерево глубины 3 умещается на экран, и его решение можно прочитать словами
tree_small = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
tree_small.fit(X_train, y_train)

plt.figure(figsize=(14, 5))
plot_tree(tree_small, feature_names=data.feature_names,
          class_names=["злокачественная", "доброкачественная"],
          filled=True, rounded=True, impurity=False, proportion=True, fontsize=8)
plt.show()

print(f"accuracy на тесте у дерева глубины 3: {tree_small.score(X_test, y_test):.4f}")

Дерево без ограничений запомнило обучающую выборку целиком, а на тесте оказалось не лучше дерева глубины 3. Значит, глубину нужно ограничивать — вопрос только в том, каким числом.

### Вопрос

Дерево глубины 3 даёт на тесте примерно столько же, сколько дерево без ограничений, но в нём три уровня вопросов вместо шести. Почему для практики это не «одинаковый результат», а явное преимущество?

_Ваш ответ:_



---
## 3. Как глубина влияет на качество

Переберём глубину от 1 до 20 и посмотрим на две кривые: качество на обучении и качество на кросс-валидации.

In [ ]:
depths = range(1, 21)
train_scores, cv_scores = [], []

for d in depths:
    model = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    train_scores.append(model.score(X_train, y_train))
    cv_scores.append(cross_val_score(model, X_train, y_train, cv=cv).mean())

best_depth_plot = list(depths)[int(np.argmax(cv_scores))]

plt.figure(figsize=(9, 4.5))
plt.plot(depths, train_scores, marker="o", ms=4, label="обучение", color="#2B5CE6")
plt.plot(depths, cv_scores, marker="D", ms=4, label="кросс-валидация", color="#C0393B")
plt.axvline(best_depth_plot, ls="--", color="#6B7484")
plt.text(best_depth_plot + 0.2, 0.9, f"лучшая глубина ≈ {best_depth_plot}", color="#6B7484")
plt.xlabel("max_depth")
plt.ylabel("доля верных ответов")
plt.xticks(list(depths))
plt.legend()
plt.title("Обучение растёт до единицы, кросс-валидация — нет")
plt.show()

print(f"лучшее качество на кросс-валидации: {max(cv_scores):.4f} при глубине {best_depth_plot}")
print(f"качество на обучении при той же глубине: {train_scores[best_depth_plot - 1]:.4f}")

**Так выглядит переобучение.** Синяя кривая доходит до единицы и там остаётся: дерево выучивает обучающую выборку наизусть. Красная перестаёт расти гораздо раньше — всё, что дерево «выучивает» после этой точки, на новых данных не работает.

Обе кривые становятся плоскими после шестого уровня: расти дальше дереву просто некуда, все листья уже чистые. Поэтому `max_depth=10` и `max_depth=None` здесь дают один и тот же результат.

Ориентироваться нужно на красную кривую. Но глубина — не единственная ручка, и перебирать параметры по одному неудобно: они влияют друг на друга.

---
## 4. Подбор параметров через GridSearchCV

`GridSearchCV` перебирает все комбинации из сетки, каждую оценивает по кросс-валидации и запоминает лучшую. Параметры дерева, которые имеет смысл перебирать:

| Параметр | Что ограничивает |
|---|---|
| `max_depth` | сколько вопросов подряд можно задать |
| `min_samples_leaf` | сколько объектов обязано остаться в листе |
| `min_samples_split` | при каком размере узла его ещё можно делить |
| `max_features` | сколько признаков рассматривать при разбиении |

Бюджет считается просто: число комбинаций умножается на число фолдов.

In [ ]:
# Пример на одном параметре: 5 значений × 5 фолдов = 25 обучений
demo = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid={"max_depth": [2, 3, 5, 8, None]},
    cv=cv, scoring="roc_auc",
)
demo.fit(X_train, y_train)

print(f"лучшие параметры: {demo.best_params_}")
print(f"лучший ROC-AUC на кросс-валидации: {demo.best_score_:.4f}")
print()
print(pd.DataFrame({
    "max_depth": [str(p["max_depth"]) for p in demo.cv_results_["params"]],
    "ROC-AUC": demo.cv_results_["mean_test_score"].round(4),
    "std": demo.cv_results_["std_test_score"].round(4),
}).to_string(index=False))

### Задание

Подберите параметры дерева по сетке и проверьте, что получилось.

1. Соберите `GridSearchCV` по двум параметрам: `max_depth` со значениями `[2, 3, 4, 5, 6, 8, 10, None]` и `min_samples_leaf` со значениями `[1, 3, 5, 10, 20]`
2. Метрика — `roc_auc`, разбиение — уже готовый `cv`
3. Обучите поиск на **обучающей** выборке и посмотрите, что он выбрал
4. Сравните с деревом по умолчанию на той же кросс-валидации

Сохраните: `best_params` — найденные параметры, `auc_tuned` — лучший ROC-AUC на кросс-валидации, `auc_default` — ROC-AUC дерева по умолчанию на кросс-валидации.

> **Что посмотреть:** `GridSearchCV(модель, param_grid=словарь, cv=cv, scoring="roc_auc", n_jobs=-1)`, после `fit` у него есть `best_params_` и `best_score_`. Качество модели по умолчанию считается через `cross_val_score(модель, X, y, cv=cv, scoring="roc_auc").mean()`.

In [ ]:
param_grid = {
    "max_depth": [2, 3, 4, 5, 6, 8, 10, None],
    "min_samples_leaf": [1, 3, 5, 10, 20],
}

# ── ВАШ КОД (три строки) ──────────────────────────────────────────
# 1. Соберите поиск по сетке и обучите его на X_train, y_train
search = ...
# 2. Заберите лучшие параметры и лучшее качество
best_params = ...
auc_tuned = ...
# ──────────────────────────────────────────────────────────────────

auc_default = cross_val_score(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    X_train, y_train, cv=cv, scoring="roc_auc",
).mean()

print(f"перебрано комбинаций: {len(param_grid['max_depth']) * len(param_grid['min_samples_leaf'])}")
print(f"лучшие параметры: {best_params}")
print()
print(f"ROC-AUC дерева по умолчанию: {auc_default:.4f}")
print(f"ROC-AUC настроенного дерева: {auc_tuned:.4f}")
print(f"прирост:                     {auc_tuned - auc_default:+.4f}")


In [ ]:
# Проверка задания
assert isinstance(best_params, dict), "best_params должен быть словарём параметров"
assert round(float(auc_tuned), 2) == 0.97, f"auc_tuned = {auc_tuned}"
assert auc_tuned > auc_default, "настроенное дерево должно обойти дерево по умолчанию"
print("Задание — верно ✓")

---
## 5. Когда параметров становится больше

Полный перебор удобен, пока параметров два. Добавим третий и четвёртый — и сетка разрастается: число комбинаций перемножается.

`RandomizedSearchCV` работает иначе: берёт `n_iter` случайных точек из заданных распределений. Бюджет задаётся напрямую и не зависит от того, сколько параметров в словаре.

In [ ]:
space = {
    "max_depth": randint(2, 21),            # не список, а распределение
    "min_samples_leaf": randint(1, 31),
    "min_samples_split": randint(2, 41),
    "max_features": [None, "sqrt", "log2"],
}

random_search = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_distributions=space, n_iter=10, cv=cv,
    scoring="roc_auc", random_state=RANDOM_STATE, n_jobs=-1,
)
random_search.fit(X_train, y_train)

full_grid = 19 * 30 * 39 * 3          # сколько точек в этом пространстве всего

print(f"{'метод':22} {'обучений':>9} {'ROC-AUC':>9}")
print(f"{'полный перебор (2 пар.)':22} {40 * 5:9} {auc_tuned:9.4f}")
print(f"{'случайный поиск (4 пар.)':22} {10 * 5:9} {random_search.best_score_:9.4f}")
print()
print(f"лучшие параметры случайного поиска: {random_search.best_params_}")
print()
print(f"Полный перебор этого же пространства занял бы {full_grid:,} комбинаций")
print(f"и {full_grid * 5:,} обучений вместо пятидесяти.")

Случайный поиск отстал на тысячные доли ROC-AUC, потратив в четыре раза меньше обучений — и при этом искал в пространстве из четырёх параметров, а не двух.

Так и выбирают между ними: пока параметров два-три и обучение быстрое, берут полный перебор, он даёт гарантию, что лучшая точка сетки найдена. Как только параметров становится больше, перебор упирается в бюджет, а случайный поиск продолжает работать: ему всё равно, сколько измерений у пространства.

### Подбор по шагам

Есть и третий способ, которым чаще всего работают руками: добавлять параметры постепенно. Сначала подбирают самый влиятельный, фиксируют найденное значение, затем берутся за следующий. На каждом шаге перебирается одна короткая сетка вместо произведения всех.

In [ ]:
tree = lambda **kw: DecisionTreeClassifier(random_state=RANDOM_STATE, **kw)
spent = 0

# Шаг 1. Самая влиятельная ручка — глубина
step1 = GridSearchCV(tree(), {"max_depth": [2, 3, 4, 5, 6, 8, 10, None]},
                     cv=cv, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
spent += len(step1.cv_results_["params"]) * 5
print(f"шаг 1 — глубина:      {step1.best_params_}, ROC-AUC {step1.best_score_:.4f}")

# Шаг 2. Глубину фиксируем, подбираем размер листа
depth = step1.best_params_["max_depth"]
step2 = GridSearchCV(tree(max_depth=depth),
                     {"min_samples_leaf": [1, 3, 5, 10, 15, 20, 30]},
                     cv=cv, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
spent += len(step2.cv_results_["params"]) * 5
leaf = step2.best_params_["min_samples_leaf"]
print(f"шаг 2 — размер листа: {step2.best_params_}, ROC-AUC {step2.best_score_:.4f}")

# Шаг 3. Уточняем обе ручки рядом с найденным
base = depth or 6
step3 = GridSearchCV(tree(), {
        "max_depth": [max(2, base - 1), base, base + 1, base + 2],
        "min_samples_leaf": [max(1, leaf - 5), leaf, leaf + 5]},
    cv=cv, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
spent += len(step3.cv_results_["params"]) * 5
print(f"шаг 3 — уточнение:    {step3.best_params_}, ROC-AUC {step3.best_score_:.4f}")

print()
print(f"{'способ':26} {'обучений':>9} {'ROC-AUC':>9}")
print(f"{'полный перебор сетки':26} {40 * 5:9} {auc_tuned:9.4f}")
print(f"{'случайный поиск':26} {10 * 5:9} {random_search.best_score_:9.4f}")
print(f"{'по шагам (три захода)':26} {spent:9} {step3.best_score_:9.4f}")

Поэтапный подбор уложился в меньший бюджет и дал результат не хуже полного перебора. Третий шаг при этом ничего не добавил — так тоже бывает, и это сигнал, что настройка закончена.

**Ловушка этого способа.** Параметры влияют друг на друга: лучшая глубина при `min_samples_leaf=1` и при `min_samples_leaf=30` может оказаться разной, а мы зафиксировали глубину на первом шаге и больше к ней не возвращались. Поэтому подбор по шагам — быстрый способ попасть в хорошую область, а не гарантия оптимума. Когда бюджет позволяет, найденное стоит перепроверить случайным поиском по всему пространству сразу.

---
## 6. Оценка качества: одной accuracy мало

У нас есть настроенная модель. Теперь — единственное обращение к тесту.

In [ ]:
best_model = search.best_estimator_
best_model.fit(X_train, y_train)

proba = best_model.predict_proba(X_test)[:, 1]
pred = best_model.predict(X_test)

proba_default = tree_default.predict_proba(X_test)[:, 1]
pred_default = tree_default.predict(X_test)

print(f"{'метрика':10} {'дерево по умолчанию':>20} {'настроенное':>13}")
print(f"{'accuracy':10} {accuracy_score(y_test, pred_default):20.4f} "
      f"{accuracy_score(y_test, pred):13.4f}")
print(f"{'ROC-AUC':10} {roc_auc_score(y_test, proba_default):20.4f} "
      f"{roc_auc_score(y_test, proba):13.4f}")
print(f"{'PR-AUC':10} {average_precision_score(y_test, proba_default):20.4f} "
      f"{average_precision_score(y_test, proba):13.4f}")

print()
print(classification_report(y_test, pred, target_names=["злокачественная", "доброкачественная"], digits=3))

In [ ]:
cm = confusion_matrix(y_test, pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["предск. злок.", "предск. доброк."],
            yticklabels=["истина злок.", "истина доброк."], ax=ax)
ax.set_title("Матрица ошибок настроенного дерева")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"пропущено злокачественных: {fp}")
print()
print("В медицинской задаче цена этих ошибок разная, поэтому смотреть")
print("нужно не на общую долю верных ответов, а на пропуски опасного класса.")

**Почему accuracy недостаточно.** Классы здесь распределены как 63 к 37 — сильного перекоса нет, поэтому accuracy выглядит прилично даже у дерева по умолчанию, хотя по ROC-AUC разрыв с настроенным деревом составляет шесть процентных пунктов. ROC-AUC и PR-AUC считаются по вероятностям и показывают, насколько уверенно модель разделяет классы, а матрица ошибок говорит, какой именно ценой достигнут результат.

---
## 7. Короткая врезка: соседи и масштаб

Дерево сравнивает признаки с порогами по одному, поэтому масштаб ему безразличен. Методу ближайших соседей — нет: он считает расстояние сразу по всем признакам.

In [ ]:
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_scaled = Pipeline([("scale", StandardScaler()),
                       ("knn", KNeighborsClassifier(n_neighbors=5))])

for name, model in [("без масштабирования", knn_raw), ("с масштабированием", knn_scaled)]:
    score = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc").mean()
    print(f"{name:22} ROC-AUC {score:.4f}")

tree_score = cross_val_score(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    X_train, y_train, cv=cv, scoring="roc_auc").mean()
print(f"{'дерево (для сравнения)':22} ROC-AUC {tree_score:.4f}")
print()
print("Масштабирование подняло соседей на три процентных пункта, а дереву оно")
print("не дало бы ничего: признаки сравниваются с порогами по одному.")

---
## Итоги занятия

- Дерево без ограничений доходит до единицы на обучении и останавливается в росте на новых данных — это переобучение видно по двум кривым
- Глубина и минимальный размер листа — главные ручки; перебирать их нужно вместе, а не по одной
- `GridSearchCV` перебирает сетку и оценивает каждую комбинацию кросс-валидацией; бюджет равен числу комбинаций, умноженному на число фолдов
- `RandomizedSearchCV` берёт заданное число случайных точек, и бюджет не зависит от числа параметров — на широких пространствах выбирают его
- **Подбор идёт только на обучающих данных.** Тест открывается один раз, когда модель уже выбрана
- Одной accuracy мало: ROC-AUC и PR-AUC считаются по вероятностям, а матрица ошибок показывает цену ошибок каждого типа
- Деревьям масштабирование не нужно, методу соседей — обязательно

## Что дальше

Следующее занятие — ансамбли: случайный лес и градиентный бустинг. Там одиночное дерево становится кирпичиком, и подбор параметров, который мы разобрали сегодня, понадобится снова.